# Stage 6 — Earth Pair Sample QA

| Field | Value |
|---|---|
| **Pipeline stage** | Stage 6 — Earth pair and label generation |
| **Previous stage** | Stage 5 — Earth network QA (`notebooks/analysis/05_earth_network_qa.ipynb`) |
| **Next stage** | Stage 7 — Earth model-input construction (`notebooks/training/03_feature_engineering.ipynb`) |
| **Purpose** | Visual inspection of touching and non-touching channel-head pairs on the basin DEM. Verifies that labels are geomorphically plausible and that branch paths are rendered correctly. |
| **Inputs** | `data/results/master_dataset_reg{A,B,C}.csv` |
| **Outputs** | Sample visualisations. Nothing written to disk. |
| **Decision gate** | Informational — confirm that touching=1 pairs look geometrically coupled and touching=0 pairs do not before proceeding to CNN patch generation. |

## 0. Configuration

In [ ]:
REGIME   = "regA"   # regA | regB | regC
BASIN    = "inyo"   # which basin to sample from
N_TOUCHING    = 4   # touching pairs to show
N_NONTOUCHING = 4   # non-touching pairs to show
RANDOM_SEED   = 42

## 1. Imports and data load

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from channel_heads.io.paths import RESULTS_DIR, EXAMPLE_DEMS
from channel_heads.basin_config import get_basin_config, LOCAL_TO_PAPER_BASIN
from channel_heads.regimes import REGIMES

regime = REGIMES[REGIME]
master_path = RESULTS_DIR / f"master_dataset_{REGIME}.csv"

if not master_path.exists():
    raise FileNotFoundError(
        f"{master_path} not found — run scripts/cli/build_earth_features_regime.py --regime {REGIME}"
    )

df_all = pd.read_csv(master_path)
df = df_all[df_all['basin'] == BASIN].copy()
if df.empty:
    raise ValueError(f"No rows for basin '{BASIN}' in {master_path}. Available: {df_all['basin'].unique().tolist()}")

paper = LOCAL_TO_PAPER_BASIN.get(BASIN, BASIN)
print(f"Regime : {REGIME}  (T={regime.threshold_km2} km²)")
print(f"Basin  : {BASIN} ({paper})")
print(f"Pairs  : {len(df)}  ({int(df['touching'].sum())} touching, {int((df['touching']==0).sum())} non-touching)")

## 2. Label distribution

In [ ]:
FEATURE_COLS = [
    'delta_L', 'orientation_diff_deg', 'headhead_dist_norm',
    'apex_angle_deg', 'proximity_profile_norm',
]

fig, axes = plt.subplots(1, len(FEATURE_COLS), figsize=(14, 3))
COLORS = {1: '#d62728', 0: '#1f77b4'}
LABELS = {1: 'touching', 0: 'non-touching'}

for ax, feat in zip(axes, FEATURE_COLS):
    if feat not in df.columns:
        ax.text(0.5, 0.5, f'{feat}\nnot found', ha='center', va='center', transform=ax.transAxes)
        continue
    for lbl in [0, 1]:
        vals = df[df['touching'] == lbl][feat].dropna()
        lo, hi = vals.quantile(0.01), vals.quantile(0.99)
        ax.hist(vals.clip(lo, hi), bins=30, alpha=0.6,
                color=COLORS[lbl], label=LABELS[lbl], density=True)
    ax.set_title(feat, fontsize=8)
    ax.set_yticks([])

axes[0].legend(fontsize=7)
fig.suptitle(f'{paper} ({REGIME}) — feature distributions by label', fontsize=9)
fig.tight_layout()
plt.show()

## 3. Load network for path visualisation

In [ ]:
from channel_heads.features.earth_enrichment import default_stream_loader
from channel_heads.units import km2_to_cells

dem_path = EXAMPLE_DEMS.get(BASIN)
cfg = get_basin_config(BASIN)
lat  = cfg['lat']
z_th = cfg['z_th']

threshold_cells = km2_to_cells(regime.threshold_km2, lat)

result = default_stream_loader(BASIN, lat, z_th, threshold_cells)
if result is None:
    raise RuntimeError(f"Could not load stream network for basin '{BASIN}'. Check DEM path.")

s, dem = result
grid_shape = dem.shape if hasattr(dem, 'shape') else dem.z.shape
print(f"Stream loaded. Grid shape: {grid_shape}")

## 4. Visualise sampled pairs on DEM

In [ ]:
from channel_heads.rasterization.earth_patches import rasterize_outlet_pair

rng = np.random.default_rng(RANDOM_SEED)

touch    = df[df['touching'] == 1].sample(min(N_TOUCHING,    len(df[df['touching'] == 1])),    random_state=RANDOM_SEED)
nontouch = df[df['touching'] == 0].sample(min(N_NONTOUCHING, len(df[df['touching'] == 0])), random_state=RANDOM_SEED)
sample   = pd.concat([touch, nontouch]).reset_index(drop=True)

CLASS_COLORS = {
    0: (0.85, 0.85, 0.85),  # background
    1: (0.20, 0.47, 0.71),  # branch A
    2: (0.89, 0.10, 0.11),  # branch B
    3: (0.30, 0.69, 0.29),  # confluence
    4: (1.00, 0.50, 0.00),  # junction overlap
}

ncols  = N_TOUCHING + N_NONTOUCHING
ngroups = 2
fig, axes = plt.subplots(1, ncols, figsize=(3 * ncols, 3.5))

for idx, (_, row) in enumerate(sample.iterrows()):
    ax = axes[idx]
    label = int(row['touching'])
    try:
        patch = rasterize_outlet_pair(
            s, int(row['outlet']), int(row['head_1']), int(row['head_2']),
            int(row['confluence']), grid_shape, target_size=128,
        )
        rgb = np.zeros((*patch.shape, 3), dtype=float)
        for cls, color in CLASS_COLORS.items():
            mask = patch == cls
            for ch, val in enumerate(color):
                rgb[..., ch][mask] = val
        ax.imshow(rgb, interpolation='nearest')
    except Exception as exc:
        ax.text(0.5, 0.5, f'error:\n{exc}', ha='center', va='center',
                transform=ax.transAxes, fontsize=6, color='red')

    border_color = '#d62728' if label else '#1f77b4'
    for spine in ax.spines.values():
        spine.set_edgecolor(border_color)
        spine.set_linewidth(3)
    ax.set_title(
        ("TOUCHING" if label else "NON-TOUCHING") +
        f"\noutlet={int(row['outlet'])}",
        fontsize=7,
        color=border_color,
    )
    ax.axis('off')

# Legend
patches = [
    mpatches.Patch(color=COLOR, label=LBL)
    for CLASS, (COLOR, LBL) in {
        1: (CLASS_COLORS[1], 'branch A'),
        2: (CLASS_COLORS[2], 'branch B'),
        3: (CLASS_COLORS[3], 'confluence'),
        4: (CLASS_COLORS[4], 'junction'),
        0: (CLASS_COLORS[0], 'background'),
    }.items()
]
fig.legend(handles=patches, loc='lower center', ncol=5, fontsize=7,
           bbox_to_anchor=(0.5, -0.05))
fig.suptitle(
    f"{paper} ({REGIME}) — red border = touching, blue = non-touching",
    fontsize=9, y=1.01
)
fig.tight_layout()
plt.show()

## 5. QC flag inspection

In [ ]:
if 'qc_flags' in df.columns:
    flagged = df[df['qc_flags'].notna() & (df['qc_flags'].astype(str) != '')]
    print(f"Pairs with QC flags: {len(flagged)}/{len(df)} ({100*len(flagged)/len(df):.1f}%)")
    if not flagged.empty:
        print("\nTop flag values:")
        print(flagged['qc_flags'].value_counts().head(10).to_string())
else:
    print("No qc_flags column in dataset.")